In [1]:
# import pandas as pd
# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import StandardScaler

# df = pd.read_csv("../train/cscada_labeled.csv") 

# # print(df.columns)

# feature_cols = [
#     'packet_count', 'total_bytes',
#        'mean_packet_size', 'std_packet_size', 'iat_mean', 'iat_std',
#        'unique_func_codes', 'exception_count'
# ]

# X = df[feature_cols].fillna(0) #to remove NaNs
# y = df["is_attack"]

# # Optional: scale features
# scaler = StandardScaler()
# X_scaled = scaler.fit_transform(X)

# X_train, X_test, y_train, y_test = train_test_split(
#     X_scaled, y, test_size=0.2, random_state=42, stratify=y
# )

# print("Train size:", X_train.shape)
# print("Test size:", X_test.shape)

# #we now split the data into training data and test data

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("../train/cscada_labeled.csv")

feature_cols = [
    'packet_count', 'total_bytes',
    'mean_packet_size', 'std_packet_size',
    'iat_mean', 'iat_std',
    'unique_func_codes', 'exception_count'
]

X = df[feature_cols].fillna(0)
y = df["is_attack"]

# First split: 80% train, 20% temp (val + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Second split: 10% val, 10% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

# Scale AFTER splitting (important to avoid data leakage)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)




Train: (989948, 8)
Validation: (123744, 8)
Test: (123744, 8)


In [2]:
#supervised models: Logistic regression (lightweight, apparently)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
print("Logistic Regression 1:")
print(classification_report(y_test, y_pred_lr))

Logistic Regression 1:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    123720
           1       0.00      0.00      0.00        24

    accuracy                           1.00    123744
   macro avg       0.50      0.50      0.50    123744
weighted avg       1.00      1.00      1.00    123744



/media/akaezist/Windows-SSD/Users/samvk/Documents/UGRC_stuff/ugrc/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/media/akaezist/Windows-SSD/Users/samvk/Documents/UGRC_stuff/ugrc/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/media/akaezist/Windows-SSD/Users/samvk/Documents/UGRC_stuff/ugrc/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predic

In [3]:
#supervised models: Logistic regression (lightweight, apparently)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

lr = LogisticRegression(max_iter=1000, class_weight='balanced')
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
print("Logistic Regression 2:")
print(classification_report(y_test, y_pred_lr))

Logistic Regression 2:
              precision    recall  f1-score   support

           0       1.00      0.48      0.65    123720
           1       0.00      1.00      0.00        24

    accuracy                           0.48    123744
   macro avg       0.50      0.74      0.32    123744
weighted avg       1.00      0.48      0.65    123744



In [4]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("Random Forest 1:")
print(classification_report(y_test, y_pred_rf))


Random Forest 1:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    123720
           1       0.00      0.00      0.00        24

    accuracy                           1.00    123744
   macro avg       0.50      0.50      0.50    123744
weighted avg       1.00      1.00      1.00    123744



In [5]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("Random Forest 2:")
print(classification_report(y_test, y_pred_rf))


Random Forest 2:
              precision    recall  f1-score   support

           0       1.00      0.81      0.90    123720
           1       0.00      0.17      0.00        24

    accuracy                           0.81    123744
   macro avg       0.50      0.49      0.45    123744
weighted avg       1.00      0.81      0.90    123744



In [6]:
print("Random Forest attempt 3:")
y_probs = rf.predict_proba(X_test)[:, 1]

for thresh in [0.5, 0.6, 0.7, 0.8]:
    y_pred = (y_probs > thresh).astype(int)
    print(f"\nThreshold: {thresh}")
    print(classification_report(y_test, y_pred))


Random Forest attempt 3:

Threshold: 0.5
              precision    recall  f1-score   support

           0       1.00      0.81      0.90    123720
           1       0.00      0.17      0.00        24

    accuracy                           0.81    123744
   macro avg       0.50      0.49      0.45    123744
weighted avg       1.00      0.81      0.90    123744


Threshold: 0.6
              precision    recall  f1-score   support

           0       1.00      0.89      0.94    123720
           1       0.00      0.08      0.00        24

    accuracy                           0.89    123744
   macro avg       0.50      0.49      0.47    123744
weighted avg       1.00      0.89      0.94    123744


Threshold: 0.7
              precision    recall  f1-score   support

           0       1.00      0.93      0.97    123720
           1       0.00      0.04      0.00        24

    accuracy                           0.93    123744
   macro avg       0.50      0.49      0.48    123744
w

In [7]:
#isolation forest, but no benign dataset used
from sklearn.ensemble import IsolationForest

X_train_benign = X_train[y_train == 0]

iso = IsolationForest(contamination=0.18, random_state=42)
iso.fit(X_train_benign)

y_pred_iso = iso.predict(X_test)

# Convert:
# -1 = anomaly → attack (1)
# 1 = normal → benign (0)
y_pred_iso = [1 if x == -1 else 0 for x in y_pred_iso]

print("Isolation Forest 1:")
print(classification_report(y_test, y_pred_iso))


Isolation Forest 1:
              precision    recall  f1-score   support

           0       1.00      0.82      0.90    123720
           1       0.00      0.17      0.00        24

    accuracy                           0.82    123744
   macro avg       0.50      0.49      0.45    123744
weighted avg       1.00      0.82      0.90    123744



In [8]:
#this is trained on benign data and then tested on attack
benign_df = pd.read_csv("../train/benign_flows.csv")

X_benign = benign_df[feature_cols].fillna(0)
X_benign_scaled = scaler.fit_transform(X_benign)

iso = IsolationForest(contamination=0.18, random_state=42)
iso.fit(X_benign_scaled)

comp_df = pd.read_csv("../train/cscada_labeled.csv")

X_comp = comp_df[feature_cols].fillna(0)
X_comp_scaled = scaler.transform(X_comp)

y_true = comp_df["is_attack"]

y_pred = iso.predict(X_comp_scaled)
y_pred = [1 if x == -1 else 0 for x in y_pred]

print("Isolation forest 2:")
print(classification_report(y_true, y_pred))



Isolation forest 2:
              precision    recall  f1-score   support

           0       1.00      0.43      0.60   1237194
           1       0.00      0.51      0.00       242

    accuracy                           0.43   1237436
   macro avg       0.50      0.47      0.30   1237436
weighted avg       1.00      0.43      0.60   1237436

